In [1]:
import cv2
import numpy as np
import os
import random
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

In [2]:
path = '/content/digits_dataset'

In [3]:
for split in ['train', 'val']:
    for i in range(10):
        os.makedirs(f'{path}/{split}/{i}', exist_ok=True)

In [4]:
def create_img(digit, save_path):

    bg_color = random.choice([0, 255])
    fg_color = 255 - bg_color
    img = np.ones((32, 32), dtype=np.uint8) * bg_color

    scale = random.uniform(0.9, 1.6)
    th = random.randint(1, 3)

    x = random.randint(-2, 12)
    y = random.randint(24, 36)

    cv2.putText(img, str(digit), (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, (fg_color,), th, cv2.LINE_AA)

    noise = np.random.randint(0, 50, (32, 32), dtype=np.uint8)
    if bg_color == 255:
        img = cv2.subtract(img, noise)
    else:
        img = cv2.add(img, noise)

    cv2.imwrite(save_path, img)

In [5]:
for d in range(10):
    for i in range(800):
        create_img(d, f'{path}/train/{d}/{i}.jpg')
    for i in range(200):
        create_img(d, f'{path}/val/{d}/{i}.jpg')

In [6]:
transform = T.Compose([
    T.Grayscale(),
    T.RandomAffine(degrees=8, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    T.Resize((32, 32)),
    T.ToTensor()
])
dataset = ImageFolder(f'{path}/train', transform=transform)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [7]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.fc = nn.Linear(32 * 8 * 8, 10)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [8]:
model = SimpleCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [9]:
for epoch in range(5):
    for images, labels in loader:
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()

In [10]:
torch.save(model.cpu().state_dict(), '/content/cnn_digits.pt')